# Results
Run the benchmarks first (see README). GNN cells skip automatically until `eval_all.csv` exists.

In [ ]:
import pathlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
R = pathlib.Path('../benchmarks/results') if pathlib.Path('../benchmarks/results').exists() else pathlib.Path('benchmarks/results')

## Baseline iteration counts (Poisson)

In [ ]:
b = pd.read_csv(R/'day1_baselines.csv')
cols = [c for c in ['none','jacobi','ic0','fsai'] if c in b]
ax = b.set_index('grid')[cols].plot.bar(figsize=(7,3.5), rot=0)
ax.set_ylabel('CG iterations'); plt.tight_layout(); plt.show()
b

## Wider pattern (classical FSAI)

In [ ]:
p = pd.read_csv(R/'day2_fsai_pattern.csv'); p

## Loss vs iterations (stand-in study with FSAI-family preconditioners)

In [ ]:
d = pd.read_csv(R/'correlation_study.csv')
none = d[d.method=='none'].set_index('matrix').iters
d['rel_iters'] = d.apply(lambda r: r.iters/none[r.matrix], axis=1)
d = d[(d.method!='none')].dropna(subset=['loss','kaporin'])
fig, ax = plt.subplots(1,2,figsize=(10,3.8))
for a,col in zip(ax,['loss','kaporin']):
    for m,g in d.groupby('method'): a.scatter(g[col], g.rel_iters, s=14, label=m)
    a.set_xscale('log'); a.set_yscale('log'); a.set_xlabel(col); a.set_ylabel('iterations / unpreconditioned')
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()
pd.read_csv(R/'correlation_summary.csv', index_col=0)

## GNN vs baselines (after training)

In [ ]:
f = R/'eval_all.csv'
if f.exists():
    e = pd.read_csv(f)
    t = e.pivot_table(index='matrix', columns='method', values='iters')
    display(t) if 'display' in globals() else print(t)
    t.plot.bar(figsize=(11,4)); plt.ylabel('CG iterations'); plt.tight_layout(); plt.show()
else:
    print('eval_all.csv not found - train a model and run scripts/evaluate_checkpoint.py')

In [ ]:
f = R/'eval_curves.npz'
if f.exists():
    c = np.load(f)
    for k in c.files: plt.semilogy(c[k], label=k)
    plt.xlabel('iteration'); plt.ylabel('relative residual'); plt.legend(fontsize=7); plt.title('Poisson 64x64'); plt.show()
else:
    print('eval_curves.npz not found')